In [1]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import pickle
load_dotenv()

with open("bfl_store.pkl", "rb") as f:
    store = pickle.load(f)

/Users/rahultiwari/Documents/02_Freelancing/Rbyte_project/Sathish_AI/test-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)


In [17]:
GOOD_PROMPT = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

In [18]:
chunk_vectors = store['vectors']
chunk_texts = store['text']
chunk_meta = store['meta']
model_name = store['model']

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5116.76it/s]


In [20]:
def retrieve_similar_chunks(query, top_k=3, threshold=0.5):
     # Convert the query into an embedding
    query_vector = model.encode([query], convert_to_numpy=True)

    # Compare query with all chunk vectors using cosine similarity
    scores = cosine_similarity(query_vector, chunk_vectors)[0]

    # filter based on thresold
    # scores

    # Get indexes of top matching chunks
    top_indexes = scores.argsort()[::-1][:top_k]

    # Return (score, index)
    results = []
    for i in top_indexes:
        results.append((float(scores[i]), int(i)))

    return results


In [21]:
def get_context(query, top_k=3):
    results = retrieve_similar_chunks(query, top_k=top_k)
  
    context_parts = []
    for score, idx in results:
        text = chunk_texts[idx]
        context_parts.append(text)

    print(context_parts)

    context = "\n\n".join(context_parts)
    return context

In [22]:
q_relevant = "what is interest rate for personal loan?"
results = get_context(q_relevant,top_k=5)

['employment category (salaried or self-employed), and the loan amount requested. All rates quoted\nbelow are effective from April 2025.\nSalaried applicants with a CIBIL score of 750 or above receive the most favourable interest rates,\nranging from 11 percent to 13 percent per annum. These applicants may borrow up to Rs 40 lakhs for a', '16 percent per annum with a maximum loan of Rs 25 lakhs. Self-employed applicants in this bracket are\ncharged between 14 and 17.5 percent per annum and may borrow up to Rs 15 lakhs for up to 48\nmonths.\nApplicants with a CIBIL score in the 650 to 699 range are placed in the highest risk bracket. Salaried', 'from 3 months to 24 months, with customers having the option of bullet repayment (principal and\ninterest paid together at maturity) or monthly interest payments with principal at maturity.\nInterest rates for gold loans range from 9.5 to 18 percent per annum. High-value loans above Rs 5 lakhs', "borrower turns 70 years of age.\nInterest rates a

In [23]:
chain = GOOD_PROMPT | llm | StrOutputParser()

In [24]:
chain.invoke({
    "context": results,
    "question": "what is interest rate for personal loan?"
})

"Interest rates for personal loans vary based on the applicant's employment category and CIBIL score. Salaried applicants with a CIBIL score of 750 or above receive rates from 11 to 13 percent per annum, while self-employed applicants in the same score bracket are charged between 14 and 17.5 percent per annum. For those with a CIBIL score between 700 and 749, salaried borrowers are offered rates from 13 to 16 percent per annum."

In [25]:
print(results)

employment category (salaried or self-employed), and the loan amount requested. All rates quoted
below are effective from April 2025.
Salaried applicants with a CIBIL score of 750 or above receive the most favourable interest rates,
ranging from 11 percent to 13 percent per annum. These applicants may borrow up to Rs 40 lakhs for a

16 percent per annum with a maximum loan of Rs 25 lakhs. Self-employed applicants in this bracket are
charged between 14 and 17.5 percent per annum and may borrow up to Rs 15 lakhs for up to 48
months.
Applicants with a CIBIL score in the 650 to 699 range are placed in the highest risk bracket. Salaried

from 3 months to 24 months, with customers having the option of bullet repayment (principal and
interest paid together at maturity) or monthly interest payments with principal at maturity.
Interest rates for gold loans range from 9.5 to 18 percent per annum. High-value loans above Rs 5 lakhs

borrower turns 70 years of age.
Interest rates are offered in two